# 🧠 MNIST MLP Classifier — Colab Edition
**Goal:** ≥ 95% test accuracy using a pure MLP (no CNNs).  
**Stack:** PyTorch · torchvision · matplotlib · seaborn  

### Model Architecture
```
Input (784) → Linear(512)+BN+ReLU+Drop → Linear(256)+BN+ReLU+Drop
           → Linear(128)+BN+ReLU+Drop → Output(10)
```
- Optimiser : Adam (lr=1e-3, weight_decay=1e-4)
- Scheduler : ReduceLROnPlateau
- Regularisation : Dropout + BatchNorm
- Auto-retries if accuracy < 80%

> **Runtime:** Enable GPU via *Runtime → Change runtime type → T4 GPU*

In [ ]:
# Cell 0 — Install any missing packages (sklearn usually pre-installed on Colab)
!pip install -q scikit-learn seaborn

In [ ]:
# Cell 1 — Imports & reproducibility
import os, time, random, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

import torchvision
import torchvision.transforms as transforms
from sklearn.metrics import confusion_matrix, classification_report

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# Cell 2 — Hyperparameters
CONFIG = {
    'batch_size'   : 128,
    'lr'           : 1e-3,
    'weight_decay' : 1e-4,
    'epochs'       : 20,
    'dropout'      : 0.3,
    'hidden_dims'  : [512, 256, 128],
    'val_fraction' : 0.1,
    'patience'     : 5,
    'min_accuracy' : 0.80,
    'target_acc'   : 0.95,
}
print('Config:', CONFIG)

In [ ]:
# Cell 3 — Data loading
def get_dataloaders(batch_size, val_fraction):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    train_full = torchvision.datasets.MNIST(root='./data', train=True,
                                            download=True, transform=transform)
    test_ds    = torchvision.datasets.MNIST(root='./data', train=False,
                                            download=True, transform=transform)
    val_size   = int(len(train_full) * val_fraction)
    train_size = len(train_full) - val_size
    train_ds, val_ds = random_split(
        train_full, [train_size, val_size],
        generator=torch.Generator().manual_seed(SEED))

    kw = dict(num_workers=2, pin_memory=True)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  **kw)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, **kw)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, **kw)

    print(f'Train: {train_size:,}  Val: {val_size:,}  Test: {len(test_ds):,}')
    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = get_dataloaders(
    CONFIG['batch_size'], CONFIG['val_fraction'])

In [ ]:
# Cell 4 — Model definition
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dim, dropout):
        super().__init__()
        layers, prev = [], input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h),
                       nn.ReLU(inplace=True), nn.Dropout(p=dropout)]
            prev = h
        layers.append(nn.Linear(prev, output_dim))
        self.net = nn.Sequential(*layers)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x.view(x.size(0), -1))

def build_model(cfg):
    m = MLP(784, cfg['hidden_dims'], 10, cfg['dropout']).to(DEVICE)
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'Architecture : 784 → {cfg["hidden_dims"]} → 10   ({n:,} params)')
    return m

In [ ]:
# Cell 5 — Training helpers
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = correct = total = 0
    with torch.set_grad_enabled(is_train):
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out  = model(imgs)
            loss = criterion(out, labels)
            if is_train:
                optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()
            total_loss += loss.item() * labels.size(0)
            correct    += out.argmax(1).eq(labels).sum().item()
            total      += labels.size(0)
    return total_loss/total, correct/total

def evaluate_test(model, loader):
    model.eval()
    all_p, all_l = [], []
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            p = model(imgs).argmax(1)
            all_p.extend(p.cpu().numpy()); all_l.extend(labels.cpu().numpy())
            correct += p.eq(labels).sum().item(); total += labels.size(0)
    return correct/total, np.array(all_p), np.array(all_l)

In [ ]:
# Cell 6 - Training loop
def train(cfg, train_loader, val_loader):
    model     = build_model(cfg)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=cfg['lr'],
                           weight_decay=cfg['weight_decay'])
    # NOTE: verbose kwarg removed in PyTorch 2.x
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3)

    history = defaultdict(list)
    best_val, best_state, no_improve = 0.0, None, 0
    prev_lr = cfg['lr']

    print(f"{'Epoch':>5} {'TrainLoss':>10} {'TrainAcc':>9} {'ValLoss':>8} {'ValAcc':>8} {'LR':>10}")
    print('-'*60)
    t0 = time.time()
    for epoch in range(1, cfg['epochs']+1):
        tl, ta = run_epoch(model, train_loader, criterion, optimizer)
        vl, va = run_epoch(model, val_loader,   criterion)
        scheduler.step(va)
        cur_lr = optimizer.param_groups[0]['lr']
        for k, v in zip(['train_loss','train_acc','val_loss','val_acc'], [tl, ta, vl, va]):
            history[k].append(v)
        lr_tag = f"{cur_lr:.2e}" + (" ↓" if cur_lr < prev_lr else "")
        prev_lr = cur_lr
        print(f"{epoch:>5} {tl:>10.4f} {ta*100:>8.2f}% {vl:>8.4f} {va*100:>7.2f}% {lr_tag:>10}")
        if va > best_val:
            best_val   = va
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= cfg['patience']:
                print(f'Early stop at epoch {epoch}'); break
    elapsed = time.time() - t0
    print(f'Done in {elapsed:.1f}s  |  Best val {best_val*100:.2f}%')
    model.load_state_dict(best_state)
    return model, history, best_val


In [ ]:
# Cell 7 — Visualisation helpers
def plot_curves(history, attempt):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13,4))
    ep = range(1, len(history['train_acc'])+1)
    ax1.plot(ep, [a*100 for a in history['train_acc']], 'b-o', ms=4, label='Train')
    ax1.plot(ep, [a*100 for a in history['val_acc']],   'r-o', ms=4, label='Val')
    ax1.axhline(95, color='green', ls='--', lw=1, label='95% target')
    ax1.set(title=f'Accuracy (Attempt {attempt})', xlabel='Epoch', ylabel='Acc (%)')
    ax1.legend(); ax1.grid(alpha=0.3); ax1.set_ylim([50,101])
    ax2.plot(ep, history['train_loss'], 'b-o', ms=4, label='Train')
    ax2.plot(ep, history['val_loss'],   'r-o', ms=4, label='Val')
    ax2.set(title=f'Loss (Attempt {attempt})', xlabel='Epoch', ylabel='Loss')
    ax2.legend(); ax2.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(f'curves_{attempt}.png', dpi=120); plt.show()

def plot_cm(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(9,7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=range(10), yticklabels=range(10))
    plt.title('Confusion Matrix — MNIST MLP', fontsize=14, fontweight='bold')
    plt.xlabel('Predicted'); plt.ylabel('True')
    plt.tight_layout(); plt.savefig('confusion_matrix.png', dpi=120); plt.show()

def show_misclassified(loader, model, n=16):
    model.eval()
    wi, wp, wt = [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            preds = model(imgs).argmax(1)
            mask  = preds != labels
            wi.extend(imgs[mask].cpu()); wp.extend(preds[mask].cpu().numpy())
            wt.extend(labels[mask].cpu().numpy())
            if len(wi) >= n: break
    fig, axes = plt.subplots(2, 8, figsize=(16,4))
    for i, ax in enumerate(axes.flat):
        if i >= min(n, len(wi)): ax.axis('off'); continue
        ax.imshow(wi[i].squeeze().numpy()*0.3081+0.1307, cmap='gray')
        ax.set_title(f'T:{wt[i]} P:{wp[i]}', color='red', fontsize=8)
        ax.axis('off')
    plt.suptitle('Misclassified (True → Predicted)', fontweight='bold')
    plt.tight_layout(); plt.savefig('misclassified.png', dpi=120); plt.show()

In [ ]:
# Cell 8 — MAIN: Iterative training loop (auto-retries if < 80%)
cfg     = dict(CONFIG)
attempt = 0
MAX_ATTEMPTS = 3

while attempt < MAX_ATTEMPTS:
    attempt += 1
    print(f'\n{"="*55}')
    print(f'  TRAINING ATTEMPT {attempt}/{MAX_ATTEMPTS}')
    print(f'  lr={cfg["lr"]:.2e} | dropout={cfg["dropout"]} | hidden={cfg["hidden_dims"]}')
    print(f'{"="*55}')

    model, history, best_val = train(cfg, train_loader, val_loader)
    plot_curves(history, attempt)

    test_acc, preds, true_labels = evaluate_test(model, test_loader)
    print(f'\n{"★"*55}')
    print(f'  🎯  TEST ACCURACY : {test_acc*100:.2f}%')
    print(f'{"★"*55}\n')

    if test_acc >= cfg['min_accuracy']:
        break

    print(f'⚠️  {test_acc*100:.2f}% < 80% — adjusting hyperparams …')
    if attempt == 1:
        cfg.update({'lr':5e-4,'dropout':0.2,'hidden_dims':[1024,512,256,128],'epochs':25})
    elif attempt == 2:
        cfg.update({'lr':2e-4,'dropout':0.1,'hidden_dims':[1024,512,256],'weight_decay':5e-5,'epochs':30})

# ── Final verdict ──────────────────────────────────────────
print(f'\n{"="*55}')
if test_acc >= cfg['target_acc']:
    print(f'  ✅  SUCCESS — {test_acc*100:.2f}%  ≥ 95% TARGET!')
elif test_acc >= cfg['min_accuracy']:
    print(f'  ✅  SUCCESS — {test_acc*100:.2f}%  ≥ 80% minimum.')
else:
    print(f'  ❌  WARNING: {test_acc*100:.2f}% did not meet 80% threshold.')
print(f'{"="*55}\n')
print(classification_report(true_labels, preds, target_names=[str(i) for i in range(10)]))

In [ ]:
# Cell 9 — Bonus: Confusion matrix + misclassified samples
plot_cm(true_labels, preds)
show_misclassified(test_loader, model)

In [ ]:
# Cell 10 — Save model (optionally to Google Drive)
USE_DRIVE = False   # ← set True to save checkpoint to Drive

save_path = 'mnist_mlp_best.pth'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    os.makedirs('/content/drive/MyDrive/mnist_mlp/', exist_ok=True)
    save_path = '/content/drive/MyDrive/mnist_mlp/mnist_mlp_best.pth'

torch.save({'model_state_dict': model.state_dict(),
            'test_accuracy'   : test_acc,
            'config'          : cfg}, save_path)
print(f'💾 Model saved → {save_path}')